# Segmentación de SuperTuxKart con SAM 2 — demo

Prueba el modelo entrenado sobre tus propias imágenes. **No hace falta instalar nada**:
ejecuta las celdas en orden con `Shift+Enter`, o *Entorno de ejecución → Ejecutar todas*.

El modelo asigna a cada píxel una de 7 clases: `background`, `track`, `kart`, `pickup`,
`nitro`, `bomb`, `projectile`.

Es transfer learning desde Segment Anything Model 2: su image encoder queda **congelado**
y solo se entrenó una cabeza de decodificación, el **3.6%** de los parámetros.

## 1. Preparar

Descarga el código y las dependencias. La primera vez tarda un par de minutos.

In [ ]:
#@title Instalar y descargar el repositorio
REPO = 'https://github.com/ferjozsot23/sam2-transfer-learning'  #@param {type:"string"}

import os, sys, subprocess

os.environ['SAM2_BUILD_CUDA'] = '0'

if not os.path.isdir('sam2stk'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, 'sam2stk'], check=True)
os.chdir('/content/sam2stk' if os.path.isdir('/content/sam2stk') else 'sam2stk')
sys.path.insert(0, os.getcwd())

try:
    import sam2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sam2'], check=True)

import torch, sam2
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
print('modelo:', os.path.getsize('model.th') / 1e6, 'MB')

In [ ]:
#@title Cargar el modelo
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from models import load_model
from utils import CLASS_NAMES, PALETTE, label_to_color, overlay

device = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo = load_model('model.th', device=device)

ent = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
tot = sum(p.numel() for p in modelo.parameters())
print('cargado en %s | backbone %s | entrada %d' % (device, modelo.backbone, modelo.image_size))
print('entrenados %.2f M de %.2f M  (%.1f%%)' % (ent/1e6, tot/1e6, 100*ent/tot))
print()
print('La imagen entra CRUDA en [0,1]; la normalizacion va dentro del modelo.')
print('Acepta cualquier resolucion de entrada.')

In [ ]:
#@title Función de segmentación
def segmentar(ruta, mostrar=True):
    img = Image.open(ruta).convert('RGB')
    x = torch.from_numpy(np.asarray(img, np.uint8).copy())\
             .permute(2,0,1).float().div_(255.)[None].to(device)
    with torch.no_grad():
        pred = modelo(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

    if mostrar:
        rgb = np.asarray(img)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5.4))
        for a, d, t in zip(ax, [rgb, label_to_color(pred),
                                overlay(rgb.transpose(2,0,1)/255., pred)],
                           ['entrada', 'segmentacion', 'superposicion']):
            a.imshow(d); a.set_title(t, fontsize=12); a.axis('off')
        presentes = sorted(int(c) for c in np.unique(pred))
        fig.legend([plt.Rectangle((0,0),1,1, fc=PALETTE[c]/255.) for c in presentes],
                   [CLASS_NAMES[c] for c in presentes],
                   loc='lower center', ncol=7, frameon=False, fontsize=11)
        fig.suptitle(os.path.basename(ruta), fontsize=13)
        fig.tight_layout(rect=[0,.06,1,.97]); plt.show()
    return pred

print('listo')

## 2. Imágenes de ejemplo

Vienen en el repositorio. Son de circuitos que el modelo **nunca vio durante el
entrenamiento**, y una de ellas (`lighthouse_sin_mascara`) ni siquiera tiene etiqueta en el
dataset original.

In [ ]:
for f in sorted(os.listdir('ejemplos')):
    segmentar(os.path.join('ejemplos', f))

## 3. Tus propias imágenes

Ejecuta la celda y selecciona uno o varios archivos. Vale cualquier resolución.

In [ ]:
from google.colab import files
subidas = files.upload()
for nombre in subidas:
    segmentar(nombre)

## 4. Descargar las máscaras

Guarda la máscara cruda de cada imagen subida (1 canal, valores 0–6) y la descarga en un zip.

In [ ]:
os.makedirs('salida', exist_ok=True)
for nombre in subidas:
    pred = segmentar(nombre, mostrar=False)
    destino = 'salida/%s_mask.png' % os.path.splitext(nombre)[0]
    Image.fromarray(pred).save(destino)
    print(destino)

!zip -qr mascaras.zip salida && echo 'mascaras.zip creado'
files.download('mascaras.zip')

---

### Por línea de comandos

```bash
pip install -r requirements.txt
python predict.py mis_imagenes/ --out resultados/ --masks
```

### En tu propio código

```python
from models import load_model
model = load_model('model.th')
pred  = model(x).argmax(1)
```
